In [1]:
import pandas as pd

print("hello")

hello


In [2]:
import scanpy as sc

adata = sc.read_h5ad(
    "data/human_dcm_hcm_scportal_03.17.2022.h5ad",
    backed="r"
)

In [3]:
adata.shape
adata.obs
adata.var
adata.obs["disease"].value_counts()
adata.obs["cell_type_leiden0.6"].value_counts()

cell_type_leiden0.6
Cardiomyocyte_I             149565
Fibroblast_I                141725
Endothelial_I                78375
Pericyte_I                   67600
Macrophage                   54715
Endothelial_II               18394
VSMC                         18137
Lymphocyte                   16246
Endocardial                   6489
Cardiomyocyte_II              5447
Adipocyte                     5320
Activated_fibroblast          5210
Lymphatic_endothelial         5181
Endothelial_III               4538
Mast_cell                     4465
Neuronal                      4292
Cardiomyocyte_III             3457
Pericyte_II                   1704
Proliferating_macrophage      1276
Fibroblast_II                  284
Epicardial                     269
Name: count, dtype: int64

In [4]:
cm1 = adata.obs["cell_type_leiden0.6"] == "Cardiomyocyte_I"

pd.crosstab(
    adata.obs.loc[cm1, "donor_id"],
    adata.obs.loc[cm1, "disease"]
)

disease,DCM,HCM,NF
donor_id,,,
P1290,3164,0,0
P1300,2764,0,0
P1304,4643,0,0
P1358,3746,0,0
P1371,4753,0,0
P1422,0,6786,0
P1425,0,1703,0
P1430,2986,0,0
P1437,3179,0,0


In [5]:
pd.crosstab(
    adata.obs.loc[cm1, "disease"],
    adata.obs.loc[cm1, "donor_id"]
).astype(bool).sum(axis=1)

disease
DCM    11
HCM    15
NF     16
dtype: int64

In [9]:
# SAVING JUST THE CM1 GROUP
# cm1_mask = adata.obs["cell_type_leiden0.6"] == "Cardiomyocyte_I"
# adata_cm1 = adata[cm1_mask].to_memory()

# adata_cm1.write_h5ad("chaffin_CM1.h5ad")

# adata.file.close()

In [1]:
import scanpy as sc

adata = sc.read_h5ad("chaffin_CM1.h5ad")

In [2]:
print(adata.shape)
print(adata.obs["disease"].value_counts())
print(adata.obs["donor_id"].nunique())


(149565, 36601)
disease
NF     61125
HCM    53024
DCM    35416
Name: count, dtype: int64
42


In [3]:
adata.obs.head()

,biosample_id,donor_id,disease,sex,age,lvef,cell_type_leiden0.6,SubCluster,cellbender_ncount,cellbender_ngenes,cellranger_percent_mito,exon_prop,cellbender_entropy,cellranger_doublet_scores
TTCTTCCGTTCAACGT-1-0,LV_1622_2_nf,P1622,NF,male,56.0,65.0,Cardiomyocyte_I,CM-X1,15815.0,4632,0.000431,0.101871,7.503471,0.091892
CATCCACCATCTAACG-1-0,LV_1622_2_nf,P1622,NF,male,56.0,65.0,Cardiomyocyte_I,CM-HHATL,15546.0,4673,0.000504,0.110197,7.582058,0.101727
ACCCAAACAGCTAACT-1-0,LV_1622_2_nf,P1622,NF,male,56.0,65.0,Cardiomyocyte_I,CM-HHATL,14983.0,4432,0.000783,0.115119,7.466405,0.089618
AAGGAATCAACTGGTT-1-0,LV_1622_2_nf,P1622,NF,male,56.0,65.0,Cardiomyocyte_I,CM-X1,14995.0,4483,0.000719,0.115444,7.452266,0.071875
TACCCGTAGCGTGCTC-1-0,LV_1622_2_nf,P1622,NF,male,56.0,65.0,Cardiomyocyte_I,CM-HHATL,14797.0,4690,0.000463,0.113069,7.609082,0.063736


In [4]:
adata.obs.shape

(149565, 14)

In [5]:
adata.obs["donor_id"].unique()

['P1622', 'P1422', 'P1722', 'P1462', 'P1558', ..., 'P1539', 'P1726', 'P1504', 'P1472', 'P1606']
Length: 42
Categories (42, object): ['P1290', 'P1300', 'P1304', 'P1358', ..., 'P1718', 'P1722', 'P1726', 'P1735']

In [6]:
print(type(adata.X))
print(adata.X.dtype)
print(adata.layers.keys())

<class 'scipy.sparse._csr.csr_matrix'>
float32
KeysView(Layers with keys: cellranger_raw)


In [7]:
print(type(adata.layers["cellranger_raw"]))
print(adata.layers["cellranger_raw"].dtype)

print(adata.X.data[:20])
print(adata.layers["cellranger_raw"].data[:20])

<class 'scipy.sparse._csr.csr_matrix'>
float32
[ 2.  8.  2. 11.  2.  1.  2.  1.  2.  1.  4.  2.  1.  1.  1.  2.  1.  1.
  1.  4.]
[ 2.  1.  1.  4.  1.  8.  2. 11.  2.  1.  2.  1.  1.  2.  1.  4.  2.  2.
  1.  1.]


In [8]:
import numpy as np
import pandas as pd
import scipy.sparse as sp

# Get donor labels as categorical codes
donor = pd.Categorical(adata.obs["donor_id"])

In [9]:
donor

['P1622', 'P1622', 'P1622', 'P1622', 'P1622', ..., 'P1561', 'P1561', 'P1561', 'P1561', 'P1561']
Length: 149565
Categories (42, object): ['P1290', 'P1300', 'P1304', 'P1358', ..., 'P1718', 'P1722', 'P1726', 'P1735']

In [10]:
test_donor = 'P1622'
mask = (adata.obs["donor_id"] == test_donor).to_numpy()
X_donor = adata.X[mask]
counts = X_donor.sum(axis = 0 )

In [11]:
print(type(mask))
print(mask.dtype)
print(mask.sum())
print(X_donor.shape)
print(type(X_donor))

<class 'numpy.ndarray'>
bool
1291
(1291, 36601)
<class 'scipy.sparse._csr.csr_matrix'>


In [12]:
print(type(counts))
print(counts.shape)

<class 'numpy.matrix'>
(1, 36601)


In [13]:
adata.var_names.get_loc("BAG3")

18737

In [14]:
adata.var_names.get_loc("TTN")

5239

In [15]:
#PSUEDO BULK BY DONORS FOR ONLY CARDIOMYOCYTES

donor_counts = []
for donor in adata.obs["donor_id"].unique().tolist():
    mask = (adata.obs["donor_id"] == donor).to_numpy()
    X_donor = adata.X[mask]
    counts = X_donor.sum(axis=0)
    counts = np.asarray(counts).ravel()
    donor_counts.append(counts)

donor_counts = np.vstack(donor_counts)

In [16]:
donor_counts

array([[ 0.,  0.,  0., ...,  0.,  3.,  4.],
       [ 0.,  0.,  0., ...,  0., 22., 11.],
       [ 0.,  0.,  0., ...,  0., 20.,  5.],
       ...,
       [ 0.,  0.,  0., ...,  0.,  1.,  6.],
       [ 0.,  0.,  0., ...,  0., 27., 25.],
       [ 0.,  0.,  0., ...,  0., 17.,  6.]],
      shape=(42, 36601), dtype=float32)

In [17]:
from pydeseq2.dds import DeseqDataSet

In [18]:
counts_df = pd.DataFrame(
    donor_counts, 
    index = adata.obs["donor_id"].unique().tolist(),
    columns = adata.var_names
)


In [19]:
metadata_df = adata.obs[["donor_id", "disease"]].drop_duplicates()

In [20]:
metadata_df = metadata_df.set_index("donor_id")

In [21]:
counts_df.index

Index(['P1622', 'P1422', 'P1722', 'P1462', 'P1558', 'P1540', 'P1718', 'P1602',
       'P1430', 'P1630', 'P1516', 'P1515', 'P1603', 'P1447', 'P1678', 'P1547',
       'P1610', 'P1510', 'P1371', 'P1479', 'P1300', 'P1617', 'P1290', 'P1600',
       'P1437', 'P1685', 'P1582', 'P1702', 'P1561', 'P1358', 'P1549', 'P1735',
       'P1304', 'P1707', 'P1425', 'P1508', 'P1631', 'P1539', 'P1726', 'P1504',
       'P1472', 'P1606'],
      dtype='object')

In [22]:
metadata_df = metadata_df.loc[counts_df.index]

In [23]:
dds = DeseqDataSet(
    counts = counts_df,
    metadata = metadata_df, 
    design = "~ disease"
)
dds.deseq2()

Using None as control genes, passed at DeseqDataSet initialization


Fitting size factors...
... done in 0.04 seconds.

Fitting dispersions...
... done in 28.59 seconds.

Fitting dispersion trend curve...
... done in 0.66 seconds.

Fitting MAP dispersions...
... done in 39.11 seconds.

Fitting LFCs...
... done in 18.59 seconds.

Calculating cook's distance...
... done in 0.06 seconds.

Replacing 168 outlier genes.

Fitting dispersions...
... done in 0.26 seconds.

Fitting MAP dispersions...
... done in 0.28 seconds.

Fitting LFCs...
... done in 0.22 seconds.



In [24]:
from pydeseq2.ds import DeseqStats

stat_res = DeseqStats(
    dds,
    contrast=["disease", "DCM", "NF"]
)

stat_res.summary()

Running Wald tests...


Log2 fold change & Wald test p-value: disease DCM vs NF
              baseMean  log2FoldChange     lfcSE      stat    pvalue      padj
MIR1302-2HG   0.030464       -0.730850  3.897158 -0.187534  0.851242       NaN
FAM138A       0.000000             NaN       NaN       NaN       NaN       NaN
OR4F5         0.000000             NaN       NaN       NaN       NaN       NaN
AL627309.1   55.287169       -0.385561  0.153469 -2.512305  0.011995  0.054217
AL627309.3    0.000000             NaN       NaN       NaN       NaN       NaN
...                ...             ...       ...       ...       ...       ...
AC141272.1    0.000000             NaN       NaN       NaN       NaN       NaN
AC023491.2    0.040872       -0.288209  3.885350 -0.074178  0.940868       NaN
AC007325.1    0.135848       -1.261262  2.838509 -0.444339  0.656797       NaN
AC007325.4   15.460732       -0.014271  0.319448 -0.044675  0.964366  0.983659
AC007325.2   15.871324       -0.697406  0.301124 -2.316011  0.020558  0.080

... done in 6.25 seconds.



In [25]:
HCM_stat_res = DeseqStats(
    dds,
    contrast=["disease", "HCM", "NF"]
)

HCM_stat_res.summary()

Running Wald tests...


Log2 fold change & Wald test p-value: disease HCM vs NF
              baseMean  log2FoldChange     lfcSE      stat    pvalue      padj
MIR1302-2HG   0.030464       -0.431214  3.568519 -0.120838  0.903819       NaN
FAM138A       0.000000             NaN       NaN       NaN       NaN       NaN
OR4F5         0.000000             NaN       NaN       NaN       NaN       NaN
AL627309.1   55.287169       -0.088595  0.140744 -0.629476  0.529037  0.692216
AL627309.3    0.000000             NaN       NaN       NaN       NaN       NaN
...                ...             ...       ...       ...       ...       ...
AC141272.1    0.000000             NaN       NaN       NaN       NaN       NaN
AC023491.2    0.040872       -0.058519  3.567393 -0.016404  0.986912       NaN
AC007325.1    0.135848       -0.384543  2.542241 -0.151261  0.879770       NaN
AC007325.4   15.460732        0.270081  0.293631  0.919800  0.357677  0.542756
AC007325.2   15.871324       -1.046598  0.283728 -3.688737  0.000225  0.002

... done in 6.30 seconds.



In [28]:
stat_res.results_df.to_csv("DCM_output.csv")
HCM_stat_res.results_df.to_csv("HCM_output.csv")

In [32]:
dcm_matrix = stat_res.results_df

sig = dcm_matrix[
    (dcm_matrix["padj"] < 0.05) &
    (dcm_matrix["log2FoldChange"].abs() > 1)
]

print("Significant:", len(sig))
print("Up:", (sig["log2FoldChange"] > 1).sum())
print("Down:", (sig["log2FoldChange"] < -1).sum())

print("\nTotal genes:", len(dcm_matrix))
print("padj available:", dcm_matrix["padj"].notna().sum())
print("padj < 0.05:", (dcm_matrix["padj"] < 0.05).sum())

Significant: 2025
Up: 861
Down: 1164

Total genes: 36601
padj available: 26182
padj < 0.05: 5639


In [39]:
dcm_matrix.loc["BAG3"]

baseMean          752.582170
log2FoldChange     -0.126762
lfcSE               0.131088
stat               -0.966997
pvalue              0.333546
padj                0.549757
Name: BAG3, dtype: float64

In [41]:
dcm_matrix.loc["TTN"]

baseMean          494220.680005
log2FoldChange         0.078553
lfcSE                  0.104210
stat                   0.753795
pvalue                 0.450973
padj                   0.656293
Name: TTN, dtype: float64

In [43]:
dcm_matrix.loc["HSF1"]

baseMean          1284.079120
log2FoldChange      -0.133474
lfcSE                0.126790
stat                -1.052714
pvalue               0.292472
padj                 0.505379
Name: HSF1, dtype: float64

In [47]:
adata.columns

AttributeError: 'AnnData' object has no attribute 'columns'

In [46]:
sig.head()

,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj
AL645608.7,15.581923,-2.638128,0.684389,-3.854721,1.158617e-04,1.486277e-03
PRDM16-DT,219.376685,-1.168683,0.233872,-4.997107,5.819674e-07,1.953471e-05
PRDM16,6382.885967,-1.008341,0.129370,-7.794234,6.480011e-15,2.976485e-12
MEGF6,40.877544,-1.085622,0.394181,-2.754118,5.885050e-03,3.198721e-02
LINC01134,46.600450,-1.739838,0.323153,-5.383939,7.287329e-08,3.431598e-06


In [35]:
sig.loc["PRDM16"]

baseMean          6.382886e+03
log2FoldChange   -1.008341e+00
lfcSE             1.293701e-01
stat             -7.794234e+00
pvalue            6.480011e-15
padj              2.976485e-12
Name: PRDM16, dtype: float64

In [37]:
"PRDM16" in sig.index

True